# Anderson Impurity Model: star and chain geometries

Walkthrough of `AndersonImpurity` (Task 21) -- one impurity orbital coupled to a bath of non-interacting levels, in two independently-specified geometries: **star** (impurity coupled directly to every bath mode) and **chain** (impurity coupled only to the first of a nearest-neighbor-hopping chain). There is no conversion between them here (the star-to-chain Lanczos mapping is out of scope) -- both are built directly from their own parameters, exactly like `FermiHubbard1D` is built directly, not converted from some other representation.

Parameters below are small synthetic numbers for demonstration -- not the material-derived values from the source document (which are confidential and stay out of the repository).

In [1]:
import numpy as np
from scipy.linalg import expm

from qaravan.core.fermions import FermionicEncoding, majorana_bilinear
from qaravan.core.hamiltonians import AndersonImpurity, bath_interleaved_mode_order

np.set_printoptions(precision=4, suppress=True)

## 1. Star geometry

$H = \varepsilon_d\sum_\sigma d_\sigma^\dagger d_\sigma + U n_{d\uparrow}n_{d\downarrow} + \sum_{k,\sigma}\varepsilon_k c_{k\sigma}^\dagger c_{k\sigma} + \sum_{k,\sigma}(V_k c_{k\sigma}^\dagger d_\sigma + \text{h.c.})$ -- the impurity couples directly to every bath mode, bath modes never couple to each other. `AndersonImpurity` is bare (a `FermionOperator`) until `.encode(...)` is called, exactly like `FermiHubbard1D`.

In [2]:
eps_d, U = -0.6, 1.6
eps_bath = [0.9, -0.4, 0.2]
V = [0.5, 0.35, 0.2]

star = AndersonImpurity.star(eps_d=eps_d, U=U, eps_bath=eps_bath, V=V)
print(f"topology={star.topology!r}  num_bath={star.num_bath}  num_modes={star.num_modes}")
print(f"{len(star.terms)} FermionTerms -- no qubits yet")

encoding = FermionicEncoding("jordan_wigner", num_modes=star.num_modes)  # identity mode_order
star_ham = star.encode(encoding)
H_star = star_ham.as_matrix()
print("matrix shape:", H_star.shape)
assert np.allclose(H_star, H_star.conj().T)
print("Hermitian: OK")

topology='star'  num_bath=3  num_modes=8
21 FermionTerms -- no qubits yet
matrix shape: (256, 256)
Hermitian: OK


## 2. Chain geometry

$H = H_{\rm imp} + \sum_\sigma\left[V(f_{0,\sigma}^\dagger d_\sigma + \text{h.c.}) + \sum_n\left(\tilde\varepsilon_n f_{n,\sigma}^\dagger f_{n,\sigma} + t_n(f_{n,\sigma}^\dagger f_{n+1,\sigma}+\text{h.c.})\right)\right]$ -- the impurity couples only to the first chain site, via a single scalar $V$; chain sites hop only to their nearest neighbor. This is a *different*, independently-specified bath (not derived from the star one above) -- same recipe as `FermiHubbard1D`'s open chain, with the impurity playing the role of "site 0".

In [3]:
eps_chain = [0.5, -0.3, 0.15]
t_chain = [0.8, 0.6]          # len(eps_chain) - 1
V_chain = 0.7

chain = AndersonImpurity.chain(eps_d=eps_d, U=U, eps_chain=eps_chain, t_chain=t_chain, V=V_chain)
print(f"topology={chain.topology!r}  num_bath={chain.num_bath}  num_modes={chain.num_modes}")

chain_encoding = FermionicEncoding("jordan_wigner", num_modes=chain.num_modes)
chain_ham = chain.encode(chain_encoding)
H_chain = chain_ham.as_matrix()
assert np.allclose(H_chain, H_chain.conj().T)
print("Hermitian: OK")

topology='chain'  num_bath=3  num_modes=8
Hermitian: OK


## 3. Sanity check: the atomic limit (no bath at all)

With `num_bath=0`, both geometries degenerate to $H_{\rm imp}$ alone -- a clean, closed-form check with no bath machinery involved. The four impurity-only eigenstates ($|0\rangle,\ |\!\uparrow\rangle,\ |\!\downarrow\rangle,\ |\!\uparrow\downarrow\rangle$) have energies $\{0,\ \varepsilon_d,\ \varepsilon_d,\ 2\varepsilon_d+U\}$ exactly.

In [4]:
atomic = AndersonImpurity.star(eps_d=eps_d, U=U, eps_bath=[], V=[])
atomic_encoding = FermionicEncoding("jordan_wigner", num_modes=atomic.num_modes)
evals = np.sort(np.linalg.eigvalsh(atomic.encode(atomic_encoding).as_matrix()))
expected = np.sort([0.0, eps_d, eps_d, 2 * eps_d + U])
print("computed:", evals)
print("expected:", expected)
assert np.allclose(evals, expected, atol=1e-10)

computed: [-0.6 -0.6  0.   0.4]
expected: [-0.6 -0.6  0.   0.4]


## 4. A real downfolded material Hamiltonian (5-bath star)

The 5-bath star parameters from Table 1 of the source document ($n_b=5 \Rightarrow 2(n_b+1)=12$ qubits, matching the document's own "12-qubit system" caption -- a nice independent check that the mode-counting convention above is right).

In [5]:
# Table 1, nb=5 row (Ca2CuO2Cl2 downfolded single-band model)
U_mat = 3.232
eps_d_mat = -1.309
eps_bath_mat = [0.735, 1.587, 0.255, 0.065, -0.001]
V_mat = [0.988, 2.432, 1.513, 0.388, 0.155]

star_mat = AndersonImpurity.star(eps_d=eps_d_mat, U=U_mat, eps_bath=eps_bath_mat, V=V_mat)
print(f"num_bath={star_mat.num_bath}, num_modes={star_mat.num_modes}")
assert star_mat.num_modes == 12

star_mat_encoding = FermionicEncoding("jordan_wigner", num_modes=star_mat.num_modes)
star_mat_ham = star_mat.encode(star_mat_encoding)

num_bath=5, num_modes=12


### Ground state

12 qubits ⇒ a $4096\times4096$ dense matrix -- exact diagonalization is still trivial at this size.

In [6]:
gs_mat = star_mat_ham.ground_state()
print(type(gs_mat).__name__)
E0_mat = gs_mat.expectation(star_mat_ham.as_observable()).real
print(f"ground energy E0 = {E0_mat:.6f}")

# Cross-check against direct diagonalization
E0_direct = np.linalg.eigvalsh(star_mat_ham.as_matrix())[0]
assert np.isclose(E0_mat, E0_direct)

Statevector
ground energy E0 = -5.678950


## 5. Cross-check against the equivalent chain

Star and chain are unrelated parameterizations *unless* the chain parameters are actually
derived from the star ones -- which the source document shows how to do (§2.1, eq. 3-7):
Lanczos-tridiagonalize the diagonal bath-energy operator $\hat\varepsilon={\rm diag}(\varepsilon_k)$,
seeded by the normalized hybridization vector $v_k = V_k/V$, $V=\sqrt{\sum_k V_k^2}$. That
produces exactly the $\tilde\varepsilon_n$, $t_n$ of eq. (7)-(8).

This conversion is *not* part of `core/` (per the proposal, out of scope for the library) --
it's a one-off, ad-hoc numpy routine here, used only to generate a genuine cross-check: if it's
implemented correctly, the resulting chain must reproduce the star spectrum *exactly*, for any
$n_b$, not just the trivial $n_b=1$ case the unit tests can check without doing this derivation.

In [7]:
def lanczos_chain_params(eps_bath, V):
    """Notebook-only, ad-hoc star->chain conversion (eq. 3-7): Lanczos-tridiagonalize
    diag(eps_bath) seeded by V/||V||. Returns (eps_chain, t_chain, V_total)."""
    eps_bath = np.asarray(eps_bath, dtype=float)
    V = np.asarray(V, dtype=float)
    num_bath = len(eps_bath)
    V_total = np.sqrt(np.sum(V**2))

    f_curr = V / V_total          # f_0, eq. (3)
    f_prev = np.zeros(num_bath)
    b_prev = 0.0
    eps_chain = np.zeros(num_bath)
    t_chain = np.zeros(num_bath - 1)

    for n in range(num_bath):
        Df = eps_bath * f_curr
        a_n = f_curr @ Df
        eps_chain[n] = a_n
        if n == num_bath - 1:
            break
        r = Df - a_n * f_curr - b_prev * f_prev
        b_n = np.linalg.norm(r)
        t_chain[n] = b_n
        f_prev, f_curr = f_curr, r / b_n
        b_prev = b_n
    return eps_chain, t_chain, V_total


eps_chain_mat, t_chain_mat, V_total_mat = lanczos_chain_params(eps_bath_mat, V_mat)
print("eps_chain:", eps_chain_mat)
print("t_chain:  ", t_chain_mat)
print("V (impurity <-> f_0):", V_total_mat)

eps_chain: [1.1436 0.7299 0.6184 0.1312 0.018 ]
t_chain:   [0.5975 0.2214 0.1914 0.0336]
V (impurity <-> f_0): 3.058513691321325


In [8]:
chain_mat = AndersonImpurity.chain(
    eps_d=eps_d_mat, U=U_mat, eps_chain=eps_chain_mat.tolist(), t_chain=t_chain_mat.tolist(), V=V_total_mat
)
chain_mat_encoding = FermionicEncoding("jordan_wigner", num_modes=chain_mat.num_modes)
chain_mat_ham = chain_mat.encode(chain_mat_encoding)

E0_chain = np.linalg.eigvalsh(chain_mat_ham.as_matrix())[0]
print(f"star  E0 = {E0_direct:.10f}")
print(f"chain E0 = {E0_chain:.10f}")
assert np.isclose(E0_direct, E0_chain, atol=1e-9)

# Stronger check: the FULL spectra agree, not just the ground energy
spec_star = np.sort(np.linalg.eigvalsh(star_mat_ham.as_matrix()))
spec_chain = np.sort(np.linalg.eigvalsh(chain_mat_ham.as_matrix()))
assert np.allclose(spec_star, spec_chain, atol=1e-9)
print("Full 4096-eigenvalue spectra agree: the Lanczos-derived chain is exactly the same physics as the star.")

star  E0 = -5.6789498172
chain E0 = -5.6789498172


Full 4096-eigenvalue spectra agree: the Lanczos-derived chain is exactly the same physics as the star.


## 6. Fixed-particle-number ground state

Reuses `Hamiltonian.ground_state(fixed_particle_number=...)` unchanged from Task 20 -- nothing
AIM-specific was needed for this to work.

In [9]:
n_op = np.diag([float(bin(i).count("1")) for i in range(2**star.num_modes)])
for n_elec in [1, 2, 3]:
    sv = star_ham.ground_state(fixed_particle_number=n_elec)
    n_expect = (sv.to_array().conj() @ n_op @ sv.to_array()).real
    print(f"fixed_particle_number={n_elec} -> <N>={n_expect:.6f}")
    assert np.isclose(n_expect, n_elec)

fixed_particle_number=1 -> <N>=1.000000
fixed_particle_number=2 -> <N>=2.000000
fixed_particle_number=3 -> <N>=3.000000


/var/folders/3y/dkx5cjxs3x5gtbx3dzjdhb980000gn/T/ipykernel_26010/2675657395.py:4: RuntimeWarning: divide by zero encountered in matmul
  n_expect = (sv.to_array().conj() @ n_op @ sv.to_array()).real
/var/folders/3y/dkx5cjxs3x5gtbx3dzjdhb980000gn/T/ipykernel_26010/2675657395.py:4: RuntimeWarning: overflow encountered in matmul
  n_expect = (sv.to_array().conj() @ n_op @ sv.to_array()).real
/var/folders/3y/dkx5cjxs3x5gtbx3dzjdhb980000gn/T/ipykernel_26010/2675657395.py:4: RuntimeWarning: invalid value encountered in matmul
  n_expect = (sv.to_array().conj() @ n_op @ sv.to_array()).real


## 7. Qubit layout: baths first, impurity last

Default (`mode_order=None`) puts the impurity first, stacked by spin. `bath_interleaved_mode_order` gives the alternative qubit order `[b1up, b1down, ..., bNup, bNdown, i_up, i_down]` -- same physics, different qubit assignment (Step B, per `docs/jw_fermi_hubbard_1d.md` §1.1).

In [10]:
order = bath_interleaved_mode_order(star.num_bath)
print("mode_order:", order)

reordered_encoding = FermionicEncoding("jordan_wigner", num_modes=star.num_modes, mode_order=order)
E0_reordered = np.linalg.eigvalsh(star.encode(reordered_encoding).as_matrix())[0]
E0_default = np.linalg.eigvalsh(star_ham.as_matrix())[0]
print(f"default E0   = {E0_default:.10f}")
print(f"reordered E0 = {E0_reordered:.10f}")
assert np.isclose(E0_default, E0_reordered)

mode_order: (1, 5, 2, 6, 3, 7, 0, 4)
default E0   = -1.7397953986
reordered E0 = -1.7397953986


## 8. Trotter circuit, and the gate-fusion warning (inherited, no new code)

`AndersonImpurity.encode()` re-chunks its own Trotter-layer structure (one layer per bath index for star, since every hybridization bond shares the impurity qubit) exactly like `FermiHubbard1D` does -- and inherits the same gate-fusion warning for free.

In [11]:
import warnings

dt = 0.01
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    circ = star_ham.trotter_circuit(dt, order=2)
    fusion_warnings = [w for w in caught if "gate-fusion" in str(w.message)]
    print(f"gate-fusion warnings: {len(fusion_warnings)}")

trotter_err = np.linalg.norm(circ.to_matrix() - expm(-1j * dt * H_star), ord="fro")
print(f"Trotter-vs-exact Frobenius error at dt={dt}: {trotter_err:.2e}")

gate-fusion warnings: 1
Trotter-vs-exact Frobenius error at dt=0.01: 1.58e-06


/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: divide by zero encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: overflow encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:80: RuntimeWarning: invalid value encountered in matmul
  result = full @ result
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: divide by zero encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: overflow encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T
/Users/alam/code/qaravan-rewrite/src/qaravan/core/circuit.py:176: RuntimeWarning: invalid value encountered in matmul
  return perm_mat @ gate_full @ perm_mat.T


## 9. A Majorana observable on the impurity (the Green's function's building block)

Eq. (12) of the source document writes the impurity Green's function in terms of correlators
of $\gamma_\sigma,\bar\gamma_\sigma$ -- the impurity's own Majorana pair. Computing the actual
time-dependent circuit (§3.1) is future work, but the observable itself is already exactly
`MajoranaTerm`/`majorana_observable` from Task 20 -- nothing AIM-specific needed to define it.

In [12]:
# Majorana indices 0,1 are gamma_{d,up}, gamma-bar_{d,up} -- the impurity's up-spin Majorana pair
# (impurity_up is mode 0 under the canonical labeling, regardless of encoding).
impurity_majorana_obs = [majorana_bilinear(0, 1)]
encoded_obs = encoding.encode_majorana_terms(impurity_majorana_obs)

gs_small = star_ham.ground_state()
value = gs_small.expectation(encoded_obs)
print(f"<i*gamma_d,up*gamma-bar_d,up> in the ground state = {value:.6f}")
assert abs(value.imag) < 1e-10

<i*gamma_d,up*gamma-bar_d,up> in the ground state = -0.697282+0.000000j


## Summary

- `AndersonImpurity.star(...)` / `.chain(...)` are bare `FermionOperator`s, no qubits until `.encode(...)`, exactly like `FermiHubbard1D`.
- The two geometries are independently-specified baths, not interconvertible in `core/` -- but Lanczos-tridiagonalizing the star's bath energies (an ad-hoc, notebook-only computation, not library code) produces a chain that reproduces the star's *entire* spectrum exactly, for the real 5-bath material parameters, not just a toy case.
- Everything downstream of `.encode()` -- `ground_state()`, `fixed_particle_number`, `trotter_circuit()`, Majorana observables -- is inherited unchanged from `Hamiltonian`/Task 20; nothing AIM-specific was needed for any of it.
- `bath_interleaved_mode_order` gives an alternative qubit layout (baths first, impurity last) with no change in physics -- Step B, independent of Step A's mode-labeling convention.